<a href="https://colab.research.google.com/github/dcangundogan/airplanefailureprediction/blob/main/dae.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [37]:
BASE = "/content/drive/MyDrive/data"
DATASET = 'FD001'

TRAIN_PATH = f"{BASE}/train_{DATASET}.csv"
TEST_PATH = f"{BASE}/test_{DATASET}.csv"
RUL_PATH = f"{BASE}/RUL_{DATASET}.txt"

LATENT_DIM = 8
BATCH_SIZE = 64
NUM_EPOCHS = 100
LEARNING_RATE = 0.001
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [38]:
torch.cuda.is_available(), torch.cuda.device_count(),torch.cuda.current_device(), torch.cuda.get_device_name(0)

(True, 1, 0, 'NVIDIA A100-SXM4-40GB')

In [4]:
np.random.seed(42)
torch.manual_seed(42);

In [5]:
def load_data(train_path, test_path, rul_path):

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    rul_values = pd.read_csv(rul_path, header=None, names=['RUL'])

    return train_df, test_df, rul_values

In [6]:
def add_rul_to_train(df):
    # Get max cycle for each unit
    max_cycles = df.groupby('unit_number')['time_in_cycles'].max().reset_index()
    max_cycles.columns = ['unit_number', 'max_cycle']

    # Merge and calculate RUL
    df = df.merge(max_cycles, on='unit_number')
    df['RUL'] = df['max_cycle'] - df['time_in_cycles']
    df.drop('max_cycle', axis=1, inplace=True)

    return df

In [7]:
def preprocess_data(train_df, test_df, sensor_cols, op_setting_cols,
                    scaler_type='minmax', clip_rul=None):
    feature_cols = op_setting_cols + sensor_cols

    # Remove columns with zero variance (constant values)
    variance = train_df[feature_cols].var()
    valid_cols = variance[variance > 0.0001].index.tolist()
    print(f"Removed {len(feature_cols) - len(valid_cols)} constant columns")
    print(f"Using {len(valid_cols)} features: {valid_cols}")

    # Select scaler
    if scaler_type == 'minmax':
        scaler = MinMaxScaler(feature_range=(0, 1))
    else:
        scaler = StandardScaler()

    # Fit scaler on training data
    train_scaled = scaler.fit_transform(train_df[valid_cols])
    test_scaled = scaler.transform(test_df[valid_cols])

    # Apply RUL clipping if specified (piece-wise linear degradation)
    if clip_rul is not None and 'RUL' in train_df.columns:
        train_df['RUL'] = train_df['RUL'].clip(upper=clip_rul)

    return train_scaled, test_scaled, scaler, valid_cols


In [8]:
def create_sequences(data, sequence_length, stride=1):
    sequences = []
    for i in range(0, len(data) - sequence_length + 1, stride):
        sequences.append(data[i:i + sequence_length])
    return np.array(sequences)

In [9]:
class DeepAutoencoder(nn.Module):

    def __init__(self, input_dim, latent_dim=8, dropout_rate=0.2):
        super(DeepAutoencoder, self).__init__()

        self.input_dim = input_dim
        self.latent_dim = latent_dim

        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            nn.Linear(32, latent_dim),
            nn.ReLU()
        )

        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            nn.Linear(32, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            nn.Linear(64, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            nn.Linear(128, input_dim),
            nn.Sigmoid()  # Output in [0, 1] for MinMax scaled data
        )

    def forward(self, x):
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)
        return reconstructed

    def encode(self, x):
        """Get latent representation."""
        return self.encoder(x)

    def decode(self, z):
        """Reconstruct from latent space."""
        return self.decoder(z)

In [10]:
class VariationalAutoencoder(nn.Module):
    """
    Variational Autoencoder (VAE) for probabilistic latent representations.

    Useful for:
    - Uncertainty quantification in health monitoring
    - Generating synthetic degradation patterns
    """

    def __init__(self, input_dim, latent_dim=8, dropout_rate=0.2):
        super(VariationalAutoencoder, self).__init__()

        self.input_dim = input_dim
        self.latent_dim = latent_dim

        # Encoder
        self.encoder_base = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
        )

        # Latent space parameters
        self.fc_mu = nn.Linear(32, latent_dim)
        self.fc_logvar = nn.Linear(32, latent_dim)

        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            nn.Linear(32, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            nn.Linear(64, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            nn.Linear(128, input_dim),
            nn.Sigmoid()
        )

    def encode(self, x):
        h = self.encoder_base(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        """Reparameterization trick for backpropagation."""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        reconstructed = self.decode(z)
        return reconstructed, mu, logvar

In [11]:
def vae_loss(reconstructed, original, mu, logvar, beta=1.0):
    """
    VAE loss = Reconstruction loss + KL divergence.

    Parameters:
    -----------
    beta : float - Weight for KL divergence (beta-VAE)
    """
    # Reconstruction loss (MSE)
    recon_loss = nn.MSELoss(reduction='sum')(reconstructed, original)

    # KL divergence
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

    return recon_loss + beta * kl_loss

In [12]:
def train_autoencoder(model, train_loader, val_loader, num_epochs=100,
                      learning_rate=0.001, device='cpu', model_type='ae',
                      early_stopping_patience=10):
    """
    Train the autoencoder model.

    Parameters:
    -----------
    model : nn.Module - Autoencoder model
    train_loader : DataLoader - Training data
    val_loader : DataLoader - Validation data
    num_epochs : int - Number of training epochs
    learning_rate : float - Learning rate
    device : str - 'cpu' or 'cuda'
    model_type : str - 'ae' for standard, 'vae' for variational
    early_stopping_patience : int - Epochs to wait before early stopping

    Returns:
    --------
    model, train_losses, val_losses
    """
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                       factor=0.5, patience=5)

    if model_type == 'ae':
        criterion = nn.MSELoss()

    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state = None

    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0.0

        for batch in train_loader:
            if isinstance(batch, (list, tuple)):
                data = batch[0].to(device)
            else:
                data = batch.to(device)

            optimizer.zero_grad()

            if model_type == 'vae':
                reconstructed, mu, logvar = model(data)
                loss = vae_loss(reconstructed, data, mu, logvar)
            else:
                reconstructed = model(data)
                loss = criterion(reconstructed, data)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)
        train_losses.append(train_loss)

        # Validation phase
        model.eval()
        val_loss = 0.0

        with torch.no_grad():
            for batch in val_loader:
                if isinstance(batch, (list, tuple)):
                    data = batch[0].to(device)
                else:
                    data = batch.to(device)

                if model_type == 'vae':
                    reconstructed, mu, logvar = model(data)
                    loss = vae_loss(reconstructed, data, mu, logvar)
                else:
                    reconstructed = model(data)
                    loss = criterion(reconstructed, data)

                val_loss += loss.item()

        val_loss /= len(val_loader)
        val_losses.append(val_loss)

        # Learning rate scheduling
        scheduler.step(val_loss)

        # Early stopping check
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_model_state = model.state_dict().copy()
        else:
            patience_counter += 1

        if (epoch + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}] - "
                  f"Train Loss: {train_loss:.6f} - Val Loss: {val_loss:.6f}")

        if patience_counter >= early_stopping_patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return model, train_losses, val_losses


In [13]:
def compute_reconstruction_error(model, data_loader, device='cpu', model_type='ae'):
    """
    Compute reconstruction error for anomaly detection.

    Returns per-sample MSE which can be used as a health indicator.
    """
    model.eval()
    errors = []

    with torch.no_grad():
        for batch in data_loader:
            if isinstance(batch, (list, tuple)):
                data = batch[0].to(device)
            else:
                data = batch.to(device)

            if model_type == 'vae':
                reconstructed, _, _ = model(data)
            else:
                reconstructed = model(data)

            # Per-sample MSE
            mse = ((data - reconstructed) ** 2).mean(dim=1)
            errors.extend(mse.cpu().numpy())

    return np.array(errors)

In [14]:
def extract_latent_features(model, data_loader, device='cpu', model_type='ae'):
    """
    Extract latent features for downstream tasks (e.g., RUL prediction).
    """
    model.eval()
    latent_features = []

    with torch.no_grad():
        for batch in data_loader:
            if isinstance(batch, (list, tuple)):
                data = batch[0].to(device)
            else:
                data = batch.to(device)

            if model_type == 'vae':
                mu, _ = model.encode(data)
                latent = mu
            else:
                latent = model.encode(data)

            latent_features.append(latent.cpu().numpy())

    return np.vstack(latent_features)

In [15]:
def plot_training_history(train_losses, val_losses, save_path=None):
    """Plot training and validation loss curves."""
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label='Training Loss', alpha=0.8)
    plt.plot(val_losses, label='Validation Loss', alpha=0.8)
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Autoencoder Training History')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()

In [16]:
def plot_reconstruction_error_distribution(errors, save_path=None):
    """Plot distribution of reconstruction errors."""
    plt.figure(figsize=(10, 5))
    plt.hist(errors, bins=50, alpha=0.7, edgecolor='black')
    plt.xlabel('Reconstruction Error (MSE)')
    plt.ylabel('Frequency')
    plt.title('Distribution of Reconstruction Errors')
    plt.axvline(np.mean(errors), color='r', linestyle='--', label=f'Mean: {np.mean(errors):.4f}')
    plt.axvline(np.percentile(errors, 95), color='orange', linestyle='--',
                label=f'95th percentile: {np.percentile(errors, 95):.4f}')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()

In [17]:
def plot_latent_space(latent_features, labels=None, save_path=None):
    """
    Visualize latent space using first 2 dimensions.
    """
    plt.figure(figsize=(10, 8))

    if labels is not None:
        scatter = plt.scatter(latent_features[:, 0], latent_features[:, 1],
                             c=labels, cmap='viridis', alpha=0.5, s=10)
        plt.colorbar(scatter, label='RUL')
    else:
        plt.scatter(latent_features[:, 0], latent_features[:, 1], alpha=0.5, s=10)

    plt.xlabel('Latent Dimension 1')
    plt.ylabel('Latent Dimension 2')
    plt.title('Latent Space Visualization')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()

In [18]:
def plot_engine_degradation(train_df, model, scaler, valid_cols,
                            unit_id, device='cpu', model_type='ae', save_path=None):
    """
    Plot reconstruction error over time for a single engine unit.
    This shows the degradation pattern detected by the autoencoder.
    """
    # Get data for specific unit
    unit_data = train_df[train_df['unit_number'] == unit_id].copy()
    unit_data = unit_data.sort_values('time_in_cycles')

    # Scale features
    unit_scaled = scaler.transform(unit_data[valid_cols])

    # Create DataLoader
    tensor_data = torch.FloatTensor(unit_scaled)
    loader = DataLoader(TensorDataset(tensor_data), batch_size=32, shuffle=False)

    # Compute reconstruction errors
    errors = compute_reconstruction_error(model, loader, device, model_type)

    # Plot
    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

    cycles = unit_data['time_in_cycles'].values
    rul = unit_data['RUL'].values

    axes[0].plot(cycles, errors, 'b-', alpha=0.7)
    axes[0].set_ylabel('Reconstruction Error')
    axes[0].set_title(f'Engine Unit {unit_id} - Degradation Pattern')
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(cycles, rul, 'r-', alpha=0.7)
    axes[1].set_xlabel('Time (Cycles)')
    axes[1].set_ylabel('Remaining Useful Life')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()


In [19]:
sensor_cols = [f'sensor_measurement_{i}' for i in range(1, 22)]
op_setting_cols = ['operational_setting_1', 'operational_setting_2', 'operational_setting_3']

In [22]:
print("\n--- Loading Data ---")
train_df, test_df, rul_df = load_data(TRAIN_PATH, TEST_PATH, RUL_PATH)
print(f"Training samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")
print(f"Number of training engines: {train_df['unit_number'].nunique()}")


--- Loading Data ---
Training samples: 20631
Test samples: 13096
Number of training engines: 100


In [24]:
train_df.head()

,unit_number,time_in_cycles,operational_setting_1,operational_setting_2,operational_setting_3,sensor_measurement_1,sensor_measurement_2,sensor_measurement_3,sensor_measurement_4,sensor_measurement_5,...,sensor_measurement_18,sensor_measurement_19,sensor_measurement_20,sensor_measurement_21,sensor_measurement_22,sensor_measurement_23,sensor_measurement_24,sensor_measurement_25,sensor_measurement_26,RUL
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,2388,100.0,39.06,23.4190,NaN,NaN,NaN,NaN,NaN,191
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,2388,100.0,39.00,23.4236,NaN,NaN,NaN,NaN,NaN,190
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,2388,100.0,38.95,23.3442,NaN,NaN,NaN,NaN,NaN,189
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,2388,100.0,38.88,23.3739,NaN,NaN,NaN,NaN,NaN,188
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,2388,100.0,38.90,23.4044,NaN,NaN,NaN,NaN,NaN,187


In [26]:
test_df.head()

,unit_number,time_in_cycles,operational_setting_1,operational_setting_2,operational_setting_3,sensor_measurement_1,sensor_measurement_2,sensor_measurement_3,sensor_measurement_4,sensor_measurement_5,...,sensor_measurement_12,sensor_measurement_13,sensor_measurement_14,sensor_measurement_15,sensor_measurement_16,sensor_measurement_17,sensor_measurement_18,sensor_measurement_19,sensor_measurement_20,sensor_measurement_21
0,1,1,0.0023,0.0003,100.0,518.67,643.02,1585.29,1398.21,14.62,...,521.72,2388.03,8125.55,8.4052,0.03,392,2388,100.0,38.86,23.3735
1,1,2,-0.0027,-0.0003,100.0,518.67,641.71,1588.45,1395.42,14.62,...,522.16,2388.06,8139.62,8.3803,0.03,393,2388,100.0,39.02,23.3916
2,1,3,0.0003,0.0001,100.0,518.67,642.46,1586.94,1401.34,14.62,...,521.97,2388.03,8130.10,8.4441,0.03,393,2388,100.0,39.08,23.4166
3,1,4,0.0042,0.0000,100.0,518.67,642.44,1584.12,1406.42,14.62,...,521.38,2388.05,8132.90,8.3917,0.03,391,2388,100.0,39.00,23.3737
4,1,5,0.0014,0.0000,100.0,518.67,642.51,1587.19,1401.92,14.62,...,522.15,2388.03,8129.54,8.4031,0.03,390,2388,100.0,38.99,23.4130


In [25]:
cols_to_drop = [f'sensor_measurement_{i}' for i in range(22, 27)]

train_df.drop(columns=cols_to_drop, inplace=True)
test_df.drop(columns=cols_to_drop, inplace=True)

sensor_cols = [col for col in sensor_cols if col not in cols_to_drop]

print(f"Dropped columns: {cols_to_drop}")
print(f"Updated sensor_cols: {sensor_cols}")

Dropped columns: ['sensor_measurement_22', 'sensor_measurement_23', 'sensor_measurement_24', 'sensor_measurement_25', 'sensor_measurement_26']
Updated sensor_cols: ['sensor_measurement_1', 'sensor_measurement_2', 'sensor_measurement_3', 'sensor_measurement_4', 'sensor_measurement_5', 'sensor_measurement_6', 'sensor_measurement_7', 'sensor_measurement_8', 'sensor_measurement_9', 'sensor_measurement_10', 'sensor_measurement_11', 'sensor_measurement_12', 'sensor_measurement_13', 'sensor_measurement_14', 'sensor_measurement_15', 'sensor_measurement_16', 'sensor_measurement_17', 'sensor_measurement_18', 'sensor_measurement_19', 'sensor_measurement_20', 'sensor_measurement_21']


In [27]:
train_df

,unit_number,time_in_cycles,operational_setting_1,operational_setting_2,operational_setting_3,sensor_measurement_1,sensor_measurement_2,sensor_measurement_3,sensor_measurement_4,sensor_measurement_5,...,sensor_measurement_13,sensor_measurement_14,sensor_measurement_15,sensor_measurement_16,sensor_measurement_17,sensor_measurement_18,sensor_measurement_19,sensor_measurement_20,sensor_measurement_21,RUL
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190,191
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236,190
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442,189
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739,188
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044,187
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20626,100,196,-0.0004,-0.0003,100.0,518.67,643.49,1597.98,1428.63,14.62,...,2388.26,8137.60,8.4956,0.03,397,2388,100.0,38.49,22.9735,4
20627,100,197,-0.0016,-0.0005,100.0,518.67,643.54,1604.50,1433.58,14.62,...,2388.22,8136.50,8.5139,0.03,395,2388,100.0,38.30,23.1594,3
20628,100,198,0.0004,0.0000,100.0,518.67,643.42,1602.46,1428.18,14.62,...,2388.24,8141.05,8.5646,0.03,398,2388,100.0,38.44,22.9333,2
20629,100,199,-0.0011,0.0003,100.0,518.67,643.23,1605.26,1426.53,14.62,...,2388.23,8139.29,8.5389,0.03,395,2388,100.0,38.29,23.0640,1


In [28]:
train_df = add_rul_to_train(train_df)

In [29]:
train_scaled, test_scaled, scaler, valid_cols = preprocess_data(
        train_df, test_df, sensor_cols, op_setting_cols,
        scaler_type='minmax', clip_rul=125
    )

Removed 10 constant columns
Using 14 features: ['sensor_measurement_2', 'sensor_measurement_3', 'sensor_measurement_4', 'sensor_measurement_7', 'sensor_measurement_8', 'sensor_measurement_9', 'sensor_measurement_11', 'sensor_measurement_12', 'sensor_measurement_13', 'sensor_measurement_14', 'sensor_measurement_15', 'sensor_measurement_17', 'sensor_measurement_20', 'sensor_measurement_21']


In [31]:
input_dim = train_scaled.shape[1]
print(f"Input dimension: {input_dim}")

Input dimension: 14


In [32]:
X_train, X_val = train_test_split(train_scaled, test_size=0.2, random_state=42)

In [33]:
train_tensor = torch.FloatTensor(X_train)
val_tensor = torch.FloatTensor(X_val)
test_tensor = torch.FloatTensor(test_scaled)

train_loader = DataLoader(TensorDataset(train_tensor), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(val_tensor), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(TensorDataset(test_tensor), batch_size=BATCH_SIZE, shuffle=False)

In [35]:
ae_model = DeepAutoencoder(input_dim=input_dim, latent_dim=LATENT_DIM, dropout_rate=0.2)

In [39]:
ae_model, ae_train_losses, ae_val_losses = train_autoencoder(
        ae_model, train_loader, val_loader,
        num_epochs=NUM_EPOCHS, learning_rate=LEARNING_RATE,
        device=DEVICE, model_type='ae', early_stopping_patience=15
    )

Epoch [10/100] - Train Loss: 0.004658 - Val Loss: 0.003179
Epoch [20/100] - Train Loss: 0.004255 - Val Loss: 0.003021
Epoch [30/100] - Train Loss: 0.004078 - Val Loss: 0.003080
Epoch [40/100] - Train Loss: 0.003892 - Val Loss: 0.003066
Epoch [50/100] - Train Loss: 0.003813 - Val Loss: 0.003031
Early stopping at epoch 53


In [43]:
plot_training_history(ae_train_losses, ae_val_losses,
                          save_path='/home/ae_training_history.png')
print("Saved training history plot.")

Saved training history plot.


In [44]:
full_train_loader = DataLoader(TensorDataset(torch.FloatTensor(train_scaled)),
                                   batch_size=BATCH_SIZE, shuffle=False)

In [45]:
train_errors = compute_reconstruction_error(ae_model, full_train_loader, DEVICE, 'ae')
print(f"Training reconstruction error - Mean: {np.mean(train_errors):.6f}, "
          f"Std: {np.std(train_errors):.6f}")


Training reconstruction error - Mean: 0.003145, Std: 0.001800


In [46]:
test_errors = compute_reconstruction_error(ae_model, test_loader, DEVICE, 'ae')
print(f"Test reconstruction error - Mean: {np.mean(test_errors):.6f}, "
          f"Std: {np.std(test_errors):.6f}")

Test reconstruction error - Mean: 0.002719, Std: 0.001339


In [48]:
plot_reconstruction_error_distribution(train_errors,
                                           save_path='/home/ae_error_distribution.png')

In [49]:
train_latent = extract_latent_features(ae_model, full_train_loader, DEVICE, 'ae')
print(f"Latent features shape: {train_latent.shape}")

Latent features shape: (20631, 8)


In [50]:
plot_latent_space(train_latent, labels=train_df['RUL'].values,
                      save_path='/home/ae_latent_space.png')

In [51]:
for unit_id in [1, 50, 100]:
        plot_engine_degradation(train_df, ae_model, scaler, valid_cols,
                               unit_id, DEVICE, 'ae',
                               save_path=f'/home/ae_degradation_unit_{unit_id}.png')

In [53]:
vae_model = VariationalAutoencoder(input_dim=input_dim, latent_dim=LATENT_DIM, dropout_rate=0.2)

vae_model, vae_train_losses, vae_val_losses = train_autoencoder(
        vae_model, train_loader, val_loader,
        num_epochs=NUM_EPOCHS, learning_rate=LEARNING_RATE,
        device=DEVICE, model_type='vae', early_stopping_patience=15
    )

Epoch [10/100] - Train Loss: 16.599537 - Val Loss: 15.962204
Epoch [20/100] - Train Loss: 16.498084 - Val Loss: 15.911541
Epoch [30/100] - Train Loss: 16.453312 - Val Loss: 15.888490
Epoch [40/100] - Train Loss: 16.446249 - Val Loss: 15.893507
Epoch [50/100] - Train Loss: 16.443026 - Val Loss: 15.889007
Early stopping at epoch 50


In [55]:
plot_training_history(vae_train_losses, vae_val_losses,
                          save_path='/home/vae_training_history.png')
vae_train_errors = compute_reconstruction_error(vae_model, full_train_loader, DEVICE, 'vae')
print(f"VAE Training reconstruction error - Mean: {np.mean(vae_train_errors):.6f}")

VAE Training reconstruction error - Mean: 0.018262


In [56]:
torch.save(ae_model.state_dict(), '/home/autoencoder_model.pth')
torch.save(vae_model.state_dict(), '/home/vae_model.pth')

In [57]:
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"Input features: {input_dim}")
print(f"Latent dimension: {LATENT_DIM}")
print(f"Best AE validation loss: {min(ae_val_losses):.6f}")
print(f"Best VAE validation loss: {min(vae_val_losses):.6f}")
print(f"Anomaly detection threshold (95th percentile): {np.percentile(train_errors, 95):.6f}")


SUMMARY
Input features: 14
Latent dimension: 8
Best AE validation loss: 0.002953
Best VAE validation loss: 15.879436
Anomaly detection threshold (95th percentile): 0.006230


In [59]:
ae_model, vae_model, scaler, valid_cols

(DeepAutoencoder(
   (encoder): Sequential(
     (0): Linear(in_features=14, out_features=128, bias=True)
     (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     (2): ReLU()
     (3): Dropout(p=0.2, inplace=False)
     (4): Linear(in_features=128, out_features=64, bias=True)
     (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     (6): ReLU()
     (7): Dropout(p=0.2, inplace=False)
     (8): Linear(in_features=64, out_features=32, bias=True)
     (9): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     (10): ReLU()
     (11): Dropout(p=0.2, inplace=False)
     (12): Linear(in_features=32, out_features=8, bias=True)
     (13): ReLU()
   )
   (decoder): Sequential(
     (0): Linear(in_features=8, out_features=32, bias=True)
     (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     (2): ReLU()
     (3): Dropout(p=0.2, inplace=False)
     (4): 